# Spain — Data Assembly per Plant

Disaggregates Spain's national PV generation (ESIOS) into per-plant hourly targets using pvlib clearsky simulation.

## Pipeline
1. Load plant metadata (593 plants) and national generation
2. Compute pvlib clearsky POA power for every plant
3. Disaggregate: `plant_gen_i = national_gen × (clearsky_i / Σ clearsky_all)`
4. Compute κ (clear-sky index): `κ_i = plant_gen_i / clearsky_power_i`
5. Match ERA5 files (82 plants) to plant_metadata by nearest grid point
6. Merge ERA5 features + generation targets into per-plant CSVs

## Design decisions
- **Disaggregation uses all 593 plants** — more plants = more accurate national clearsky denominator
- **Clearsky model**: pvlib Ineichen (Linke turbidity atlas; no ERA5 required for this step)
- **ERA5 available for 82 plants only** — those 82 form the TFT training set
- **κ as target**: normalises plant size and location, enables multi-plant TFT

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import pvlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) != 'spain_total':
    candidate = os.path.join(os.getcwd(), 'spain_total')
    if os.path.isdir(candidate):
        os.chdir(candidate)
print(f'Working directory: {os.getcwd()}')

In [ ]:
META_FILE  = 'data/plant_metadata.csv'
GEN_FILE   = 'data/spain_pv_generation.csv'
ERA5_DIR   = 'era5_timeseries_plants'
OUTPUT_DIR = 'data/per_plant'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TIME_START = '2023-01-01'
TIME_END   = '2025-12-31'

# ERA5-Land column rename map (applied after unit conversion)
ERA5_RENAME = {
    't2m' : 'temperature_2m_C',
    'd2m' : 'dewpoint_2m_C',
    'sp'  : 'surface_pressure_hPa',
    'tp'  : 'total_precip_mm',
    'ssrd': 'ssrd_wm2',
    'strd': 'strd_wm2',
}

## 1. Load inputs

In [ ]:
meta = pd.read_csv(META_FILE)
print(f'Plants loaded: {len(meta)}')
print(meta[['plant_id','plant_name','capacity_mw','latitude','longitude',
            'tracking_type','tilt_deg','azimuth_deg']].head())

In [ ]:
gen = pd.read_csv(GEN_FILE, index_col='datetime_utc', parse_dates=True)
gen = gen.loc[TIME_START:TIME_END]
print(f'National generation rows: {len(gen)}')
print(f'Period: {gen.index[0]} -> {gen.index[-1]}')

## 2. Pvlib clearsky POA power for all 593 plants

- **Fixed tilt**: `get_total_irradiance` with static surface_tilt / surface_azimuth from PVGIS metadata
- **Single-axis tracker**: pvlib `singleaxis` gives time-varying tilt/azimuth per hour
- **Clearsky DC proxy**: `capacity_mw × (POA_W/m² / 1000)` — sufficient for proportional disaggregation weights

Note on azimuth convention: PVGIS outputs 0° = South. pvlib uses 180° = South. We add 180° when passing to pvlib.

In [ ]:
times = gen.index  # UTC DatetimeIndex, hourly


def clearsky_poa_power(row, times):
    loc = pvlib.location.Location(
        latitude=row['latitude'],
        longitude=row['longitude'],
        altitude=0,
        tz='UTC',
    )
    solpos = loc.get_solarposition(times)
    cs     = loc.get_clearsky(times, model='ineichen')

    if row['tracking_type'] == 'single_axis':
        tracking = pvlib.tracking.singleaxis(
            apparent_zenith=solpos['apparent_zenith'],
            apparent_azimuth=solpos['azimuth'],
            axis_tilt=0,
            axis_azimuth=180,   # N-S axis (standard for Spain)
            max_angle=60,
            backtrack=True,
            gcr=0.35,
        )
        surface_tilt    = tracking['surface_tilt'].fillna(0)
        surface_azimuth = tracking['surface_azimuth'].fillna(180)
    else:
        surface_tilt    = row['tilt_deg']
        surface_azimuth = row['azimuth_deg'] + 180  # PVGIS: 0=S -> pvlib: 180=S

    poa = pvlib.irradiance.get_total_irradiance(
        surface_tilt=surface_tilt,
        surface_azimuth=surface_azimuth,
        dni=cs['dni'],
        ghi=cs['ghi'],
        dhi=cs['dhi'],
        solar_zenith=solpos['apparent_zenith'],
        solar_azimuth=solpos['azimuth'],
    )
    return row['capacity_mw'] * poa['poa_global'].clip(lower=0) / 1000.0


print('Computing clearsky for all plants (a few minutes)...')
clearsky_power = {}
for _, row in tqdm(meta.iterrows(), total=len(meta), desc='pvlib clearsky'):
    clearsky_power[row['plant_id']] = clearsky_poa_power(row, times)

clearsky_df = pd.DataFrame(clearsky_power, index=times)
print(f'Clearsky matrix shape: {clearsky_df.shape}')

In [ ]:
peak_hour    = clearsky_df.sum(axis=1).idxmax()
total_cs_peak = clearsky_df.sum(axis=1).max()
total_capacity = meta['capacity_mw'].sum()
print(f'Total installed capacity : {total_capacity:,.0f} MW')
print(f'Peak total clearsky      : {total_cs_peak:,.0f} MW  at {peak_hour}')
print(f'Peak capacity factor     : {total_cs_peak / total_capacity:.2%}')

## 3. Disaggregate national generation → per-plant targets

In [ ]:
national_cs  = clearsky_df.sum(axis=1)
safe_denom   = national_cs.replace(0, np.nan)
shares       = clearsky_df.div(safe_denom, axis=0).fillna(0)

national_gen = gen['pv_generation_mwh'].reindex(times).ffill()
plant_gen    = shares.multiply(national_gen, axis=0)

print(f'Plant generation matrix : {plant_gen.shape}')
print(f'Sum check at peak hour  : national={national_gen.loc[peak_hour]:,.1f}  '
      f'sum of plant targets={plant_gen.loc[peak_hour].sum():,.1f}  MWh')

In [ ]:
# kappa: clear-sky index = plant_gen / clearsky_power
with np.errstate(divide='ignore', invalid='ignore'):
    kappa = plant_gen.div(clearsky_df.replace(0, np.nan)).clip(0, 2).fillna(0)

daytime = clearsky_df.sum(axis=1) > 100
print('kappa stats (daytime hours):')
print(kappa[daytime].stack().describe().round(3))

## 4. Load ERA5 files and convert units

ERA5-Land unit conversions (same as Cáceres pipeline):
- `t2m`, `d2m` : K → °C (−273.15)
- `sp` : Pa → hPa (÷100)
- `tp` : m → mm (×1000)
- `ssrd`, `strd` : J/m² accumulated/hr → W/m² (÷3600)

In [ ]:
def convert_era5(df):
    df = df.copy()
    df['t2m']  = df['t2m']  - 273.15
    df['d2m']  = df['d2m']  - 273.15
    df['sp']   = df['sp']   / 100.0
    df['tp']   = df['tp']   * 1000.0
    df['ssrd'] = df['ssrd'] / 3600.0
    df['strd'] = df['strd'] / 3600.0
    df = df.rename(columns=ERA5_RENAME)
    df['dewpoint_depression_C'] = df['temperature_2m_C'] - df['dewpoint_2m_C']
    df['kt_era5'] = (df['ssrd_wm2'] / 1361).clip(0, 1)
    return df


def load_era5_file(path):
    raw   = pd.read_csv(path, parse_dates=['timestamp'])
    lat_r = round(float(raw['latitude'].iloc[0]), 1)
    lon_r = round(float(raw['longitude'].iloc[0]), 1)
    df    = raw.set_index('timestamp').sort_index().loc[TIME_START:TIME_END]
    df    = convert_era5(df[['t2m', 'd2m', 'sp', 'tp', 'ssrd', 'strd']])
    return lat_r, lon_r, df


era5_files = sorted(glob.glob(os.path.join(ERA5_DIR, 'plant_*/plant_*_combined.csv')))
print(f'ERA5 files found: {len(era5_files)}')

era5_by_gridpoint = {}
for fpath in tqdm(era5_files, desc='Loading ERA5'):
    lat_r, lon_r, df = load_era5_file(fpath)
    key = (lat_r, lon_r)
    if key not in era5_by_gridpoint:
        era5_by_gridpoint[key] = df

print(f'Unique ERA5 grid points: {len(era5_by_gridpoint)}')

In [ ]:
# Match each plant to nearest ERA5 grid point (max 0.5 deg ~ 55 km)
era5_grid = np.array(list(era5_by_gridpoint.keys()))


def nearest_era5(lat, lon):
    dists = np.sqrt((era5_grid[:, 0] - lat)**2 + (era5_grid[:, 1] - lon)**2)
    idx   = dists.argmin()
    return tuple(era5_grid[idx]) if dists[idx] <= 0.5 else None


meta['era5_key'] = meta.apply(lambda r: nearest_era5(r['latitude'], r['longitude']), axis=1)

n_with = meta['era5_key'].notna().sum()
print(f'Plants with ERA5 match : {n_with} / {len(meta)}')
print(f'Plants without ERA5    : {len(meta) - n_with}')

## 5. Export per-plant CSVs

In [ ]:
era5_feature_cols = [
    'temperature_2m_C', 'dewpoint_2m_C', 'surface_pressure_hPa',
    'total_precip_mm', 'ssrd_wm2', 'strd_wm2',
    'dewpoint_depression_C', 'kt_era5',
]

exported_with_era5    = 0
exported_without_era5 = 0

for _, row in tqdm(meta.iterrows(), total=len(meta), desc='Exporting'):
    pid = row['plant_id']

    df = pd.DataFrame(index=times)
    df.index.name = 'datetime_utc'

    df['plant_gen_mwh']     = plant_gen[pid].values
    df['clearsky_power_mw'] = clearsky_df[pid].values
    df['kappa']             = kappa[pid].values
    df['capacity_mw']       = row['capacity_mw']
    df['latitude']          = row['latitude']
    df['longitude']         = row['longitude']
    df['tilt_deg']          = row['tilt_deg']
    df['azimuth_sin']       = row['azimuth_sin']
    df['azimuth_cos']       = row['azimuth_cos']
    df['tracking_type']     = row['tracking_type']

    key = row['era5_key']
    if key is not None:
        era5 = era5_by_gridpoint[key][era5_feature_cols].reindex(times)
        df   = df.join(era5)
        exported_with_era5 += 1
    else:
        exported_without_era5 += 1

    df.to_csv(os.path.join(OUTPUT_DIR, f'{pid}.csv'))

print(f'Exported with ERA5    : {exported_with_era5}')
print(f'Exported without ERA5 : {exported_without_era5}')
print(f'Output: {OUTPUT_DIR}/')

## 6. Validation

In [ ]:
sample_pid = meta[meta['era5_key'].notna()].iloc[0]['plant_id']
sample = pd.read_csv(
    os.path.join(OUTPUT_DIR, f'{sample_pid}.csv'),
    index_col='datetime_utc', parse_dates=True,
)
print(f'Sample: {sample_pid}')
print(f'Columns ({len(sample.columns)}): {list(sample.columns)}')
print(f'Shape: {sample.shape}')
sample.head()

In [ ]:
cap = sample['capacity_mw'].iloc[0]
daytime_mask = sample['clearsky_power_mw'] > cap * 0.02
kappa_day = sample.loc[daytime_mask, 'kappa']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

kappa_day.hist(bins=50, ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title(f'kappa distribution (daytime) — {sample_pid}')
axes[0].set_xlabel('kappa (clear-sky index)')

week = sample.loc['2024-07-01':'2024-07-07']
axes[1].plot(week.index, week['clearsky_power_mw'], label='Clearsky', lw=1.2, color='orange')
axes[1].plot(week.index, week['plant_gen_mwh'],     label='Disagg. target', lw=1.2, color='steelblue')
axes[1].set_title('Generation vs clearsky — July 2024 sample')
axes[1].legend()
axes[1].set_ylabel('MW')

plt.tight_layout()
plt.show()

## Summary

| Field | Value |
|---|---|
| **Plants processed** | 593 |
| **Plants with ERA5** | ~82 |
| **Target variable** | κ (clear-sky index) = plant_gen / clearsky_power |
| **Disaggregation** | Proportional to pvlib Ineichen clearsky POA power |
| **ERA5 matching** | Nearest 0.1° grid point, max 0.5° search radius |
| **Output** | `data/per_plant/plant_XXX.csv` |

**Next:** `3_nwp_download.ipynb` — Open-Meteo NWP forecasts for all 593 plant locations.